# Construcción de variables ponderadas por población 2024

1. **Objetivo del notebook**

- Añadir versiones ponderadas por población de las variables ambientales y sus rezagos y medias móviles.

2. **Decisiones metodológicas**

- Calcular `population_weight` como la población de cada celda dividida por la población total del área de estudio.
- Aplicar la misma ponderación a las variables ambientales dinámicas y a sus derivados temporales.
- Mantener las variables originales, las claves del panel, la población y las variables de resultado sin ponderar.
- Conservar como nulos los valores faltantes.

3. **Fuente**

- `data/processed/integrated/analytical_grid_week_2024_lags_n_rolls.parquet`.

4. **Producto**

- `data/processed/integrated/analytical_grid_week_2024_lags_n_rolls_popw.parquet`.


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import HTML, display

for candidate in [Path.cwd(), *Path.cwd().resolve().parents]:
    if (candidate / 'src' / 'paths.py').is_file():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError('No se encontró src/paths.py.')

import sys
sys.path.insert(0, str(PROJECT_ROOT))
from src.plot_palette import COLORS, MAP_CMAP, apply_plot_palette
apply_plot_palette()

INPUT_PATH = PROJECT_ROOT / 'data/processed/integrated/analytical_grid_week_2024_lags_n_rolls.parquet'
OUTPUT_PATH = PROJECT_ROOT / 'data/processed/integrated/analytical_grid_week_2024_lags_n_rolls_popw.parquet'
DYNAMIC_ENVIRONMENTAL = [
    'tmean_c', 'tmin_c', 'tmax_c',
    'rh_mean', 'rh_min', 'rh_max', 'tp_sum',
    'ndvi_mean_biweekly', 'ndvi_max_biweekly',
    'ndvi_min_biweekly', 'ndvi_std_biweekly',
]

df = pd.read_parquet(INPUT_PATH).copy()
population_counts = df.groupby('grid_id')['population'].nunique(dropna=False)
if population_counts.gt(1).any():
    raise ValueError('Hay más de una población por grid_id.')
grid_population = df[['grid_id', 'population']].drop_duplicates('grid_id')
population_total = grid_population['population'].sum(min_count=1)
df = df.merge(grid_population[['grid_id']], on='grid_id', how='left', validate='many_to_one')
df['population_weight'] = df['population'] / population_total

weighted_columns = [
    column for column in df.columns
    if column in DYNAMIC_ENVIRONMENTAL
    or any(column.startswith(f'{base}_lag_') or column.startswith(f'{base}_rolling_mean_4')
           for base in DYNAMIC_ENVIRONMENTAL)
]
for column in weighted_columns:
    df[f'{column}_popw'] = df[column] * df['population_weight']

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(OUTPUT_PATH, index=False)
print(f'Parquet guardado: {OUTPUT_PATH} ({len(df):,} filas, {len(df.columns):,} columnas)')


Parquet guardado: C:\tesis_aedes_grilla_2024\data\processed\integrated\analytical_grid_week_2024_lags_n_rolls_popw.parquet (177,268 filas, 129 columnas)


## Vista del parquet generado

Primeras 10 filas del producto con variables ponderadas por población. La barra inferior permite recorrer todas las columnas.

In [2]:
preview = pd.read_parquet(OUTPUT_PATH).head(10)
display(HTML(
    '<div style="width:100%; overflow-x:auto; padding-bottom:8px;">'
    + preview.to_html(index=False, max_cols=None)
    + '</div>'
))


grid_id,year,epiweek,tmean_c,tmin_c,tmax_c,elevation_min,elevation_mean,elevation_max,rh_mean,rh_min,rh_max,tp_sum,koppen_class,population,ndvi_mean_biweekly,ndvi_max_biweekly,ndvi_min_biweekly,ndvi_std_biweekly,epiweek_sin,epiweek_cos,total_records,vector_findings,eggs,larvae,pupae,adults,confirmed_cases,reporting_centers,tmean_c_lag_1,tmean_c_lag_2,tmean_c_lag_4,tmean_c_rolling_mean_4,tmin_c_lag_1,tmin_c_lag_2,tmin_c_lag_4,tmin_c_rolling_mean_4,tmax_c_lag_1,tmax_c_lag_2,tmax_c_lag_4,tmax_c_rolling_mean_4,rh_mean_lag_1,rh_mean_lag_2,rh_mean_lag_4,rh_mean_rolling_mean_4,rh_min_lag_1,rh_min_lag_2,rh_min_lag_4,rh_min_rolling_mean_4,rh_max_lag_1,rh_max_lag_2,rh_max_lag_4,rh_max_rolling_mean_4,tp_sum_lag_1,tp_sum_lag_2,tp_sum_lag_4,tp_sum_rolling_mean_4,ndvi_mean_biweekly_lag_1,ndvi_mean_biweekly_lag_2,ndvi_mean_biweekly_lag_4,ndvi_mean_biweekly_rolling_mean_4,ndvi_max_biweekly_lag_1,ndvi_max_biweekly_lag_2,ndvi_max_biweekly_lag_4,ndvi_max_biweekly_rolling_mean_4,ndvi_min_biweekly_lag_1,ndvi_min_biweekly_lag_2,ndvi_min_biweekly_lag_4,ndvi_min_biweekly_rolling_mean_4,ndvi_std_biweekly_lag_1,ndvi_std_biweekly_lag_2,ndvi_std_biweekly_lag_4,ndvi_std_biweekly_rolling_mean_4,population_weight,tmean_c_popw,tmin_c_popw,tmax_c_popw,rh_mean_popw,rh_min_popw,rh_max_popw,tp_sum_popw,ndvi_mean_biweekly_popw,ndvi_max_biweekly_popw,ndvi_min_biweekly_popw,ndvi_std_biweekly_popw,tmean_c_lag_1_popw,tmean_c_lag_2_popw,tmean_c_lag_4_popw,tmean_c_rolling_mean_4_popw,tmin_c_lag_1_popw,tmin_c_lag_2_popw,tmin_c_lag_4_popw,tmin_c_rolling_mean_4_popw,tmax_c_lag_1_popw,tmax_c_lag_2_popw,tmax_c_lag_4_popw,tmax_c_rolling_mean_4_popw,rh_mean_lag_1_popw,rh_mean_lag_2_popw,rh_mean_lag_4_popw,rh_mean_rolling_mean_4_popw,rh_min_lag_1_popw,rh_min_lag_2_popw,rh_min_lag_4_popw,rh_min_rolling_mean_4_popw,rh_max_lag_1_popw,rh_max_lag_2_popw,rh_max_lag_4_popw,rh_max_rolling_mean_4_popw,tp_sum_lag_1_popw,tp_sum_lag_2_popw,tp_sum_lag_4_popw,tp_sum_rolling_mean_4_popw,ndvi_mean_biweekly_lag_1_popw,ndvi_mean_biweekly_lag_2_popw,ndvi_mean_biweekly_lag_4_popw,ndvi_mean_biweekly_rolling_mean_4_popw,ndvi_max_biweekly_lag_1_popw,ndvi_max_biweekly_lag_2_popw,ndvi_max_biweekly_lag_4_popw,ndvi_max_biweekly_rolling_mean_4_popw,ndvi_min_biweekly_lag_1_popw,ndvi_min_biweekly_lag_2_popw,ndvi_min_biweekly_lag_4_popw,ndvi_min_biweekly_rolling_mean_4_popw,ndvi_std_biweekly_lag_1_popw,ndvi_std_biweekly_lag_2_popw,ndvi_std_biweekly_lag_4_popw,ndvi_std_biweekly_rolling_mean_4_popw
0,2024,1,21.209301,19.992899,22.276896,25.0,33.596774,43.0,74.356583,71.468781,76.885185,0.016606,BWh,0.0,0.050019,0.081831,-0.021624,0.004055,0.120537,0.992709,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,2024,2,22.054096,21.573660,22.765982,25.0,33.596774,43.0,73.982315,71.317276,76.284805,0.011030,BWh,0.0,0.048979,0.080882,-0.004966,0.003381,0.239316,0.970942,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,21.209301,NaN,NaN,NaN,19.992899,NaN,NaN,NaN,22.276896,NaN,NaN,NaN,74.356583,NaN,NaN,NaN,71.468781,NaN,NaN,NaN,76.885185,NaN,NaN,NaN,0.016606,NaN,NaN,NaN,0.050019,NaN,NaN,NaN,0.081831,NaN,NaN,NaN,-0.021624,NaN,NaN,NaN,0.004055,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.0,0.0,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,-0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN
0,2024,3,22.568083,22.038534,23.052988,25.0,33.596774,43.0,75.665398,72.660240,78.163437,0.068585,BWh,0.0,0.047940,0.079934,0.011691,0.002707,0.354605,0.935016,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,22.054096,21.209301,NaN,NaN,21.573660,19.992899,NaN,NaN,22.765982,22.276896,NaN,NaN,73.982315,